# Module 14: Async Human Approval - 03: Multi-reviewer sign-off

> **MLCourse - Agentic AI - Agent Patterns**

Notebooks 01-02 modelled one reviewer per request - even escalation just
moves the single decision to a different person. Some decisions need more
than one signature at once: a large financial transaction needing finance
**and** engineering sign-off, a production deployment needing two engineers'
approval, a legal change needing both legal and compliance.

This is **N-of-M approval**: a single suspended thread waits not for one
answer but for a *quorum* of answers from a defined set of reviewers.

### What you will learn

1. Why one `interrupt()` cannot naturally hold "waiting on 2 of 3 people."
2. Modelling ballots as durable, independent records.
3. A quorum-checking resume loop, including the reject-fast case.
4. What happens when reviewers disagree.

Still no LLM calls, no API key, no web server.

### Setup


In [ ]:
import json
import os
import sqlite3
import time
import uuid
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Optional, TypedDict

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

QUEUE_PATH = Path("signoff_queue.json")
DB_PATH = "signoff_demo.db"
if QUEUE_PATH.exists():
    QUEUE_PATH.unlink()
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

checkpointer = SqliteSaver(sqlite3.connect(DB_PATH, check_same_thread=False))
print("multi-reviewer sign-off demo ready -- no API key needed")


### 1. Why a single `interrupt()` doesn't naturally model this

`interrupt()` suspends a node waiting for **one** `Command(resume=value)`.
The moment that arrives, the node continues - there is no built-in notion of
"received one answer, still waiting for two more." You could imagine calling
`interrupt()` three times in a loop, once per reviewer, but that forces the
reviewers to answer **in a fixed order**, one at a time, which is not how
real sign-off works - three people should be able to review in parallel and
whoever finishes first, finishes first.

The fix is the same idea notebook 01 already used for the single-reviewer
case, generalised: **the durable store, not the interrupt payload, is the
source of truth.** A ballot box holding individual reviewer votes, and a
resume loop that only fires once the ballot box shows quorum.

### 2. Ballots: independent records, not one shared answer

Each reviewer's vote is its **own** durable record, written independently, so
two reviewers voting at the same moment (from different processes, different
machines) never race each other or overwrite one another's vote.

### The ballot box


In [ ]:
@dataclass
class SignoffRequest:
    request_id: str
    thread_id: str
    summary: str
    required_reviewers: list           # WHO must vote
    quorum: int                        # HOW MANY approvals needed
    votes: dict = field(default_factory=dict)      # reviewer -> "approve"/"reject"
    reasons: dict = field(default_factory=dict)
    created_at: float = field(default_factory=time.time)
    status: str = "pending"            # pending | approved | rejected


class SignoffQueue:
    def __init__(self, path: Path):
        self.path = path
        if not self.path.exists():
            self._write({})

    def _read(self) -> dict:
        return json.loads(self.path.read_text()) if self.path.exists() else {}

    def _write(self, data: dict) -> None:
        self.path.write_text(json.dumps(data, indent=2))

    def enqueue(self, thread_id, summary, required_reviewers, quorum) -> str:
        req = SignoffRequest(request_id=str(uuid.uuid4())[:8], thread_id=thread_id,
                             summary=summary, required_reviewers=required_reviewers,
                             quorum=quorum)
        data = self._read()
        data[req.request_id] = asdict(req)
        self._write(data)
        return req.request_id

    def get(self, request_id: str) -> dict:
        return self._read()[request_id]

    def cast_vote(self, request_id: str, reviewer: str, approve: bool, reason: str = "") -> dict:
        """Record ONE reviewer's vote. Independent of every other vote --
        this is what lets reviewers vote in any order, from any process,
        without stepping on each other."""
        data = self._read()
        r = data[request_id]
        if reviewer not in r["required_reviewers"]:
            raise ValueError(f"{reviewer} is not on the required reviewer list for {request_id}")
        r["votes"][reviewer] = "approve" if approve else "reject"
        r["reasons"][reviewer] = reason
        self._recompute_status(r)
        self._write(data)
        return r

    def _recompute_status(self, r: dict) -> None:
        """Quorum logic: reject-fast (one reject ends it immediately -- a
        sign-off is not a majority vote, it is a set of gates that must ALL
        eventually pass), approve once enough YES votes are in."""
        if "reject" in r["votes"].values():
            r["status"] = "rejected"
            return
        approvals = sum(1 for v in r["votes"].values() if v == "approve")
        if approvals >= r["quorum"]:
            r["status"] = "approved"

    def pending(self) -> list:
        return [r for r in self._read().values() if r["status"] == "pending"]


queue = SignoffQueue(QUEUE_PATH)
print("ballot-box queue ready")


### The design choice worth naming: reject-fast

`_recompute_status` treats a **single** reject as decisive, regardless of
quorum. That is a deliberate modelling choice, not an accident: a sign-off
gate is usually "everyone required must be OK with this," not "a majority is
OK with this." If your domain genuinely wants majority voting instead
(3 of 5 approve wins even with 2 rejects), that is a different aggregation
rule - the ballot-box *shape* stays the same, only `_recompute_status`
changes. Get this decision right for your domain before you build on it;
getting it wrong silently changes what "approved" means.

### 3. The graph: one suspension, waiting on quorum

The node still calls `interrupt()` exactly once - it is not looping over
reviewers. What it is waiting *for* is a resume signal that only fires once
the ballot box, checked from outside, shows quorum reached.

### The graph


In [ ]:
class DeployState(TypedDict):
    change: str
    request_id: str
    verdict: str


from langgraph.runtime import get_config


def request_signoff(state: DeployState) -> dict:
    # Read the ACTUAL thread_id for this invocation from LangGraph's runtime
    # config, rather than closing over a module-level variable -- a node
    # that hard-codes or captures a thread_id only ever works for one thread.
    this_thread = get_config()["configurable"]["thread_id"]
    req_id = queue.enqueue(
        thread_id=this_thread,
        summary=f"Production deploy: {state['change']}",
        required_reviewers=["oncall-eng@company", "sre-lead@company", "eng-manager@company"],
        quorum=2,     # 2-of-3 required -- not unanimous, not just one
    )
    verdict = interrupt({"request_id": req_id})
    return {"request_id": req_id, "verdict": verdict}


builder = StateGraph(DeployState)
builder.add_node("request_signoff", request_signoff)
builder.add_edge(START, "request_signoff")
builder.add_edge("request_signoff", END)
agent = builder.compile(checkpointer=checkpointer)

thread_id = "deploy-prod-2026-08-31"
cfg = {"configurable": {"thread_id": thread_id}}
agent.invoke({"change": "migrate payments DB to new cluster", "request_id": "", "verdict": ""}, cfg)

req_id = queue.pending()[0]["request_id"]
req = queue.get(req_id)
print(f"request {req_id}: quorum {req['quorum']}-of-{len(req['required_reviewers'])}")
print(f"required reviewers: {req['required_reviewers']}")


### 4. Three reviewers vote, independently, in any order

Cast the votes as three separate calls, exactly as three different people
using three different laptops would - nothing here assumes an order.

### Independent votes


In [ ]:
print("sre-lead votes first (reviewers can arrive in any order):")
r = queue.cast_vote(req_id, "sre-lead@company", approve=True,
                    reason="Runbook looks solid.")
print(f"  votes so far: {r['votes']}  -> status: {r['status']}")

print("\noncall-eng votes second:")
r = queue.cast_vote(req_id, "oncall-eng@company", approve=True,
                    reason="Tested in staging.")
print(f"  votes so far: {r['votes']}  -> status: {r['status']}")
print(f"  (quorum of {r['quorum']} reached -- eng-manager never had to vote)")


### Resuming once quorum is reached - even with a vote outstanding

Notice: `eng-manager@company` was on the required reviewer list and never
voted, and the request is already `approved`. That is the quorum policy
working exactly as declared (2-of-3), not a bug - but it is worth being
deliberate about, because "2 of 3, and the third person's opinion is now
moot" is a real product decision, not a detail to gloss over. If every
listed reviewer's input genuinely must be captured before resuming
regardless of quorum, set `quorum` equal to the full reviewer count.

### Resuming: same pattern as notebook 01, gated on quorum


In [ ]:
def resume_if_quorum(graph, q: SignoffQueue, thread_id: str) -> Optional[dict]:
    snap = graph.get_state({"configurable": {"thread_id": thread_id}})
    if not snap.next:
        return None

    matches = [r for r in q._read().values()
              if r["thread_id"] == thread_id and r["status"] != "pending"]
    if not matches:
        return None       # still waiting on quorum

    decision = matches[0]
    return graph.invoke(Command(resume=decision["status"]),
                        {"configurable": {"thread_id": thread_id}})


result = resume_if_quorum(agent, queue, thread_id)
print(f"resumed: {result}")


### 5. The reject-fast path

Now the other branch: one required reviewer rejects. Quorum for approval was
never reached, and per the policy above, that is decisive immediately -
demonstrated on a fresh request so the passing case above stays untouched.

### A second request, this time with a rejection


In [ ]:
thread_id_2 = "deploy-prod-risky-change"
cfg2 = {"configurable": {"thread_id": thread_id_2}}
agent.invoke({"change": "enable untested feature flag globally",
             "request_id": "", "verdict": ""}, cfg2)

req_id_2 = [r["request_id"] for r in queue.pending()
           if r["thread_id"] == thread_id_2][0]

print("oncall-eng approves:")
r = queue.cast_vote(req_id_2, "oncall-eng@company", approve=True, reason="Looks fine to me.")
print(f"  status: {r['status']}")

print("\nsre-lead REJECTS:")
r = queue.cast_vote(req_id_2, "sre-lead@company", approve=False,
                    reason="No rollback plan documented.")
print(f"  status: {r['status']}  <- decisive immediately, even though quorum")
print(f"  ({r['quorum']}) was never reached by APPROVE votes")

result2 = resume_if_quorum(agent, queue, thread_id_2)
print(f"\nresumed: {result2}")


### Reading the rejection

The thread resumed with `verdict="rejected"` after exactly **one** reject,
despite only one approve vote being in and `eng-manager` never weighing in at
all. That is the reject-fast rule from section 2 doing its job: for a
production deployment, one qualified reviewer's documented objection ("no
rollback plan") is enough to stop the change, and there is no reason to wait
out the rest of the ballot to find that out.

### Key takeaways

- A single `interrupt()` cannot natively hold "waiting on N of M answers" -
  it fires once. The ballot box, checked from outside the graph, is what
  turns several independent votes into one resume decision.
- Model each reviewer's vote as its **own durable record**, so reviewers can
  vote in any order, from any process, without racing each other.
- Decide your aggregation rule deliberately: this notebook used
  **reject-fast** (any rejection is immediately decisive) plus a
  **quorum threshold** for approval - a different domain might need strict
  unanimity, or true majority voting. The ballot-box *shape* does not change;
  the `_recompute_status` rule does, and getting it wrong silently changes
  what "approved" means.
- Resuming a multi-reviewer thread uses the **same** `Command(resume=...)`
  pattern as the single-reviewer case in notebook 01 - the complexity is
  entirely in the ballot box, not in how the graph is resumed.

That completes module 14. Related reading:
[`02_langgraph/04_human_in_the_loop`](../../02_langgraph/04_human_in_the_loop/README.md)
for `interrupt()` fundamentals with a human at the keyboard, and
[`02_langgraph/10_state_migration_and_versioning`](../../02_langgraph/10_state_migration_and_versioning/README.md)
for what happens to a thread suspended this long if your state schema changes
before anyone gets around to voting.